# Die Struktur im Original aus (für kleinere ResNets wie ResNet-18 oder 34):
## Basic Block :RealResidualBlock von Kaiming He

https://arxiv.org/abs/1512.03385

Die "kleinen" ResNets (18 & 34 Schichten)
Diese verwenden den sogenannten Basic Block. Dieser besteht – wie du richtig gelesen hast – immer aus zwei aufeinanderfolgenden 
 Convolution-Schichten. 


ResNet-18: Besteht aus 8 solcher Basic Blocks (plus Eingangs- und Ausgangsschicht).

ResNet-34: Besteht aus 16 solcher Basic Blocks. Hier sieht man besonders gut, dass das Netz trotz der Tiefe besser trainiert als ein "flaches" (plain) Netz mit 34 Schichten



In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------- Residual Block (BasicBlock, 2 Convs) --------
class RealResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity     # Shortcut Addition
        out = self.relu(out)
        return out

# -------- Mini-ResNet --------
class MiniResNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2)  # 128x128 -> 64x64
        )

        # 4 Residual Blocks
        self.layer1 = RealResidualBlock(16)
        self.layer2 = RealResidualBlock(16)
        self.layer3 = RealResidualBlock(16)
        self.layer4 = RealResidualBlock(16)

        # Klassifizierung
        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.global_pool(x)      # -> [B,16,1,1]
        x = torch.flatten(x, 1)      # -> [B,16]
        x = self.fc(x)               # -> [B,num_classes]
        return x

# Testlauf

In [3]:
x = torch.randn(2, 3, 128, 128)  # Batch=2, RGB, 128x128
model = MiniResNet(num_classes=10)
y = model(x)
print("Output shape:", y.shape)

Output shape: torch.Size([2, 10])
